In [20]:
from pathlib import Path
import pandas as pd
import json
import gzip
import os

print("Libraries imported successfully")

Libraries imported successfully


In [22]:
PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
META_DIR = DATA_DIR / "metadata"

RESULTS_DIR = PROJECT_DIR / "results"
REPORTS_DIR = PROJECT_DIR / "reports"

for folder in [RAW_DIR, META_DIR, RESULTS_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw data exists:", RAW_DIR.exists())

Project: C:\Users\praja\functional-connectivity
Raw data exists: True


In [24]:
bold_files = list(RAW_DIR.rglob("*task-rest*_bold.nii.gz"))

print("Resting-state MRI files:", len(bold_files))

for file in bold_files[:10]:
    print(file.name, round(file.stat().st_size / (1024**2), 2), "MB")

Resting-state MRI files: 1
sub-NDARINVYT858CBN_task-restAP_run-01_bold.nii.gz 567.57 MB


In [26]:
PILOT_ID = "sub-NDARINVYT858CBN"

pilot_files = [
    f for f in bold_files
    if PILOT_ID in str(f)
]

for file in pilot_files:
    print(file.name)

sub-NDARINVYT858CBN_task-restAP_run-01_bold.nii.gz


In [34]:
integrity_results = []

for file in pilot_files:

    try:
        with gzip.open(file, "rb") as f:
            while f.read(1024 * 1024):
                pass

        status = "VALID"

    except Exception as e:
        status = f"FAILED: {type(e).__name__}"

    integrity_results.append({
        "filename": file.name,
        "size_MB": round(file.stat().st_size / (1024**2), 2),
        "integrity": status
    })

integrity_df = pd.DataFrame(integrity_results)

display(integrity_df)

,filename,size_MB,integrity
0,sub-NDARINVYT858CBN_task-restAP_run-01_bold.ni...,567.57,VALID


In [30]:
import gzip
from pathlib import Path

ap_file = AP_FILES[0]

print("File:", ap_file.name)
print("Size (MB):", round(ap_file.stat().st_size / (1024**2), 2))

try:
    with gzip.open(ap_file, "rb") as f:
        while f.read(1024 * 1024):
            pass

    print("GZIP integrity: PASSED")

except Exception as e:
    print("GZIP integrity: FAILED")
    print(type(e).__name__, e)

File: sub-NDARINVYT858CBN_task-restAP_run-01_bold.nii.gz
Size (MB): 567.57
GZIP integrity: PASSED


In [35]:
inventory_path = RESULTS_DIR / "milestone1_inventory.csv"

integrity_df.to_csv(inventory_path, index=False)

print("Inventory saved:", inventory_path)

Inventory saved: C:\Users\praja\functional-connectivity\results\milestone1_inventory.csv


In [38]:
summary = {
    "dataset": "ds005237",
    "version": "1.1.3",
    "participants_total": 241,
    "pilot_participant": PILOT_ID,
    "restAP_files": len(AP_FILES),
    "restPA_files": len(PA_FILES),
    "valid_files": int(
        (integrity_df["integrity"] == "VALID").sum()
    ),
    "failed_files": int(
        integrity_df["integrity"].str.startswith("FAILED").sum()
    )
}

summary_path = REPORTS_DIR / "milestone1_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print(json.dumps(summary, indent=4))

{
    "dataset": "ds005237",
    "version": "1.1.3",
    "participants_total": 241,
    "pilot_participant": "sub-NDARINVYT858CBN",
    "restAP_files": 1,
    "restPA_files": 0,
    "valid_files": 1,
    "failed_files": 0
}
